# Traning AlignmentEvaluationNN

### Step 2.1: Generate Alignment Training Data

Generate alignments with RMSD labels:
```bash
sbatch make_transformations_12A.slurm
```

**What this does:**
- For each protein complex in training set:
  - Extracts target patch (p1) and source patches (p2)
  - Performs descriptor matching
  - Aligns patches using RANSAC
  - Computes RMSD to ground truth
  - Saves alignment data

**Outputs** (per protein):
```
training_data_12A_seed_benchmark/PDBID_CHAIN1_CHAIN2/
├── aligned_source_patches.npy          # Aligned coordinates
├── aligned_source_patches_normals.npy  # Surface normals
├── aligned_source_patches_descs_0.npy  # Descriptors
├── aligned_source_patches_dists_to_target_atoms.npy  # Atom distances
├── source_patch_rmsds.npy              # RMSD labels (< 2.0 = positive)
├── target_patch.npy                    # Target patch coordinates
└── ... (more metadata)
```

**Duration:** ~1-2 hours per protein (parallelized via SLURM array)


In [ ]:
!sbatch make_transformations_12A.slurm

### Visualize examples of positive and negative alignments

The next cell loads one target directory from `training_data_12A_seed_benchmark/`,
uses `source_patch_rmsds.npy` (`rmsd < 2.0` = positive) as labels, and visualizes:

- label counts and RMSD summary
- RMSD histogram with the 2.0 A threshold
- 3D overlays of sampled positive and negative aligned patches on the target patch

In [1]:
# Visualize one aligned patch pair in context of untransformed protein structures (py3Dmol)
from pathlib import Path
import numpy as np
import py3Dmol
from Bio.PDB import PDBParser, PDBIO
from io import StringIO
import os

# Config: reuse from visualization cell above, or override
#TARGET_ID = "1AVX_A_B"
DATA_ROOT = Path("training_data_12A_seed_benchmark")
RMSD_POSITIVE_THRESHOLD = 2.0
SPHERE_RADIUS = 0.2

required_files = [
    "source_patch_rmsds.npy",
    "aligned_source_patches.npy",
    "target_patch.npy",
    "alignment_transformations.npy",
    "source_patch_names.npy",
    "aligned_source_patches_dists_to_target_atoms.npy",
]

# Resolve PDB directory (same layout as make_transformations_12A)
repo_root = Path(os.popen("git rev-parse --show-toplevel").read().strip())
PDB_DIR = repo_root / "masif" / "data" / "masif_ppi_search" / "data_preparation" / "01-benchmark_pdbs"


if not PDB_DIR.exists():
    print(f"ERROR: PDB directory does not exist: {PDB_DIR}")
else:
    candidate_dirs = sorted([p for p in DATA_ROOT.iterdir() if p.is_dir()])
    df_list = []
    for target_dir in candidate_dirs:
        missing = [fn for fn in required_files if not (target_dir / fn).exists()]
        if missing:
            continue
        rmsd = np.load(target_dir / "source_patch_rmsds.npy", allow_pickle=True)
        aligned = np.load(target_dir / "aligned_source_patches.npy", allow_pickle=True)
        target_patch = np.load(target_dir / "target_patch.npy", allow_pickle=True)
        alignment_transformations = np.load(
            target_dir / "alignment_transformations.npy", allow_pickle=True
        )
        source_patch_names = np.load(
            target_dir / "source_patch_names.npy", allow_pickle=True
        )
        aligned_source_patches_dists_to_target_atoms = np.load(
            target_dir / "aligned_source_patches_dists_to_target_atoms.npy", allow_pickle=True
        )

        n = min(len(rmsd), len(aligned), len(alignment_transformations), len(source_patch_names))
        rmsd = np.asarray(rmsd[:n], dtype=float)
        aligned = aligned[:n]
        alignment_transformations = alignment_transformations[:n]
        source_patch_names = source_patch_names[:n]
        target_center = np.mean(target_patch, axis=0)
        patch_centre_distances = []
        

        for i in range(n):
            if str(source_patch_names[i]) == target_dir.name:
                T = np.asarray(alignment_transformations[i])
                pts = np.asarray(aligned[i])
                pts_h = np.hstack([pts, np.ones((len(pts), 1))])
                src_orig = (np.linalg.inv(T) @ pts_h.T).T[:, :3]
                source_center = np.mean(src_orig, axis=0)
                patch_centre_distances.append(float(np.linalg.norm(target_center - source_center)))
            else:
                patch_centre_distances.append(np.nan)

        dists_to_target = aligned_source_patches_dists_to_target_atoms[:n]
        dist_to_target_mean = [float(np.mean(d)) for d in dists_to_target]

        df_list.append({
            "target_id": target_dir.name,
            "pair_index": np.arange(n),
            "source_patch": [str(x) for x in source_patch_names],
            "rmsd": rmsd,
            "distance": patch_centre_distances,
            "dist_to_target_mean": dist_to_target_mean,
            "label": ["pos" if r < RMSD_POSITIVE_THRESHOLD else "neg" for r in rmsd],
        })

    if df_list:
        try:
            import pandas as pd
            dfs = [pd.DataFrame(d) for d in df_list]
            df = pd.concat(dfs, ignore_index=True)
            df = df.sort_values(by=["target_id", "rmsd"])
            display(df.head())
        except ImportError:
            for d in df_list:
                print(f"--- {d['target_id']} ---")
                for j in range(len(d["rmsd"])):
                    lbl = d["label"][j]
                    dist_val = d["distance"][j]
                    d_str = f"{dist_val:.3f}" if not np.isnan(dist_val) else "nan"
                    print(f"  {d['pair_index'][j]:<6} {d['source_patch'][j]:<12} {d['rmsd'][j]:>8.3f} {d_str:>10} {lbl:>6}")
    else:
        print("No target directories with required files found.")


,target_id,pair_index,source_patch,rmsd,distance,dist_to_target_mean,label
231,1A0G_A_B,231,1A0G_A_B,0.080041,16.721630,5.731595,pos
203,1A0G_A_B,203,1A0G_A_B,0.506264,2.687313,2.293857,pos
365,1A0G_A_B,365,1A0G_A_B,0.516608,3.433596,2.444331,pos
164,1A0G_A_B,164,1A0G_A_B,0.534736,1.824826,2.287416,pos
816,1A0G_A_B,816,1A0G_A_B,0.538374,2.960043,2.450661,pos


In [2]:
# ----------- Potential issues ------------
# TODO: Check if there is any positive pair with unusually large distance

# sort the pos entries by distance
df[df['label'] == 'pos'].sort_values(by='distance', ascending=False).head(10)

,target_id,pair_index,source_patch,rmsd,distance,dist_to_target_mean,label
51712,1B9L_C_D,556,1B9L_C_D,0.219396,20.372391,4.852570,pos
25362,1APZ_B_D,164,1APZ_B_D,0.021861,19.771634,9.430471,pos
12826,1A99_C_D,541,1A99_C_D,0.272807,19.675475,9.204642,pos
54191,1BBH_A_B,772,1BBH_A_B,0.027048,19.425391,6.270654,pos
51579,1B9L_C_D,423,1B9L_C_D,0.219396,19.303849,6.378535,pos
90163,1CSG_A_B,347,1CSG_A_B,0.607929,19.165133,10.638998,pos
86273,1COZ_A_B,604,1COZ_A_B,0.876808,19.053433,3.221904,pos
51870,1B9L_C_D,714,1B9L_C_D,1.162051,18.812134,10.395159,pos
85900,1COZ_A_B,231,1COZ_A_B,0.876808,18.536270,3.271110,pos
46819,1B4U_C_D,804,1B4U_C_D,0.634148,18.495143,7.860872,pos


In [3]:
# Config: define which pair of patches to visualize
TARGET_ID = "1B9L_C_D"
SHOW_POSITIVE = False  # True = positive pair, False = negative pair (used when PAIR_INDEX is None)
PAIR_OFFSET = None  # Which pair in that category: 0 = first, 1 = second, etc.
PAIR_INDEX = 556  # If set (e.g. 5), use this index directly; overrides SHOW_POSITIVE and PAIR_OFFSET

def add_struct_to_py3dmol(structure, view=None):
    """Convert Bio.PDB Structure to PDB string and add to py3Dmol view."""
    io = PDBIO()
    io.set_structure(structure)
    pdb_buf = StringIO()
    io.save(pdb_buf)
    pdb_str = pdb_buf.getvalue()
    if view is None:
        view = py3Dmol.view(width=600, height=500)
    view.addModel(pdb_str, "pdb")
    return view


def add_patch_spheres(view, coords, color, radius=0.2):
    """Add small spheres at patch vertex coordinates to the py3Dmol view."""
    for pt in coords:
        view.addSphere({
            "center": {"x": float(pt[0]), "y": float(pt[1]), "z": float(pt[2])},
            "radius": radius,
            "color": color,
        })


if not PDB_DIR.exists():
    print(f"ERROR: PDB directory does not exist: {PDB_DIR}")
else:
    candidate_dirs = sorted([p for p in DATA_ROOT.iterdir() if p.is_dir()])
    if TARGET_ID is None:
        target_dir = candidate_dirs[0] if candidate_dirs else None
    else:
        target_dir = DATA_ROOT / TARGET_ID

    if target_dir is None or not target_dir.exists():
        print(f"ERROR: Target directory not found: {target_dir}")
    else:
        missing = [fn for fn in required_files if not (target_dir / fn).exists()]
        if missing:
            print(f"ERROR: Missing required files in {target_dir.name}: {missing}")
        else:
            rmsd = np.load(target_dir / "source_patch_rmsds.npy", allow_pickle=True)
            aligned = np.load(target_dir / "aligned_source_patches.npy", allow_pickle=True)
            target_patch = np.load(target_dir / "target_patch.npy", allow_pickle=True)
            alignment_transformations = np.load(
                target_dir / "alignment_transformations.npy", allow_pickle=True
            )
            source_patch_names = np.load(
                target_dir / "source_patch_names.npy", allow_pickle=True
            )

            n = min(len(rmsd), len(aligned), len(alignment_transformations), len(source_patch_names))
            rmsd = np.asarray(rmsd[:n], dtype=float)
            aligned = aligned[:n]
            alignment_transformations = alignment_transformations[:n]
            source_patch_names = source_patch_names[:n]

            pos_idx = np.where(rmsd < RMSD_POSITIVE_THRESHOLD)[0]
            neg_idx = np.where(rmsd >= RMSD_POSITIVE_THRESHOLD)[0]

            if PAIR_INDEX is not None:
                pair_idx = int(PAIR_INDEX) if 0 <= PAIR_INDEX < n else None
            elif SHOW_POSITIVE and len(pos_idx) > PAIR_OFFSET:
                pair_idx = pos_idx[PAIR_OFFSET]
            elif not SHOW_POSITIVE and len(neg_idx) > PAIR_OFFSET:
                pair_idx = neg_idx[PAIR_OFFSET]
            else:
                pair_idx = None

            if pair_idx is None:
                if PAIR_INDEX is not None:
                    print(f"No pair at index {PAIR_INDEX} (valid range: 0–{n-1}).")
                else:
                    cat = "positive" if SHOW_POSITIVE else "negative"
                    print(f"No {cat} pair at offset {PAIR_OFFSET} (available: {len(pos_idx)} pos, {len(neg_idx)} neg).")
            else:
                i = int(pair_idx)
                rmsd_val = rmsd[i]
                src_name = str(source_patch_names[i])
                parts = src_name.split("_")
                src_pdb_id = parts[0]
                src_chain_p2 = parts[2] if len(parts) >= 3 else parts[1]

                target_name = target_dir.name
                tgt_parts = target_name.split("_")
                tgt_pdb_id = tgt_parts[0]
                tgt_chain_p1 = tgt_parts[1] if len(tgt_parts) >= 2 else tgt_parts[0]
                tgt_chain_p2 = tgt_parts[2] if len(tgt_parts) >= 3 else tgt_parts[1]

                pdb_p1 = PDB_DIR / f"{tgt_pdb_id}_{tgt_chain_p1}.pdb"
                pdb_p2 = PDB_DIR / f"{src_pdb_id}_{src_chain_p2}.pdb"

                if not pdb_p1.exists():
                    print(f"ERROR: PDB not found: {pdb_p1}")
                elif not pdb_p2.exists():
                    print(f"ERROR: PDB not found: {pdb_p2}")
                else:
                    parser = PDBParser(QUIET=True)
                    struct_p1 = parser.get_structure("p1", str(pdb_p1))
                    struct_p2 = parser.get_structure("p2", str(pdb_p2))

                    # Inverse transform to get source patch in untransformed (source) frame
                    T = np.asarray(alignment_transformations[i])
                    aligned_pts = np.asarray(aligned[i])
                    pts_h = np.hstack([aligned_pts, np.ones((len(aligned_pts), 1))])
                    source_patch_orig = (np.linalg.inv(T) @ pts_h.T).T[:, :3]

                    view = py3Dmol.view(width=600, height=500)
                    add_struct_to_py3dmol(struct_p1, view)
                    view.setStyle({"model": 0}, {"cartoon": {"color": "cyan"}, "line": {"colorscheme": "cyanCarbon"}})
                    add_struct_to_py3dmol(struct_p2, view)
                    view.setStyle({"model": 1}, {"cartoon": {"color": "lightgreen"}, "line": {"colorscheme": "lightgreenCarbon"}})
                    add_patch_spheres(view, target_patch, "blue", radius=SPHERE_RADIUS)
                    add_patch_spheres(view, source_patch_orig, "green", radius=SPHERE_RADIUS)
                    # add the transformed source patch in lightgray
                    add_patch_spheres(view, aligned_pts, "lightgray", radius=SPHERE_RADIUS)
                    # Apply transformation to p2 PDB model and add in lightgray
                    import copy
                    struct_p2_transformed = copy.deepcopy(struct_p2)
                    for atom in struct_p2_transformed.get_atoms():
                        c = atom.get_coord()
                        c_h = np.array([c[0], c[1], c[2], 1.0])
                        c_new = (T @ c_h)[:3]
                        atom.set_coord(c_new)
                    add_struct_to_py3dmol(struct_p2_transformed, view)
                    view.setStyle({"model": 2}, {"cartoon": {"color": "lightgray"}})

                    view.zoomTo()
                    view.show()

                    is_pos = rmsd_val < RMSD_POSITIVE_THRESHOLD
                    label = "positive" if is_pos else "negative"
                    print(f"P1: PDB ID {tgt_pdb_id}, chain {tgt_chain_p1}")
                    print(f"P2: PDB ID {src_pdb_id}, chain {src_chain_p2}")
                    print(f"RMSD: {rmsd_val:.3f} Å")
                    print(f"Label: {label} (rmsd {'<' if is_pos else '>='} {RMSD_POSITIVE_THRESHOLD} Å)")


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

P1: PDB ID 1B9L, chain C
P2: PDB ID 1B9L, chain D
RMSD: 0.219 Å
Label: positive (rmsd < 2.0 Å)


In [ ]:
# Visualize all positive and negative patch pairs in the untransformed protein context (py3Dmol)
# Spheres at patch centers, lines between target and source centers (blue=positive, red=negative)
from pathlib import Path
import numpy as np
import py3Dmol
from Bio.PDB import PDBParser, PDBIO
from io import StringIO
import os

# Config: reuse from visualization cell above, or override
TARGET_ID = globals().get("TARGET_ID", None)
DATA_ROOT = Path("training_data_12A_seed_benchmark")
RMSD_POSITIVE_THRESHOLD = globals().get("RMSD_POSITIVE_THRESHOLD", 2.0)
N_MAX_POS = 100
N_MAX_NEG = 100
SPHERE_RADIUS = 0.3
RNG_SEED = 7

required_files = [
    "source_patch_rmsds.npy",
    "aligned_source_patches.npy",
    "target_patch.npy",
    "alignment_transformations.npy",
    "source_patch_names.npy",
]

repo_root = Path(os.popen("git rev-parse --show-toplevel").read().strip())
PDB_DIR = repo_root / "masif" / "data" / "masif_ppi_search" / "data_preparation" / "01-benchmark_pdbs"


def add_struct_to_py3dmol(structure, view=None):
    """Convert Bio.PDB Structure to PDB string and add to py3Dmol view."""
    io = PDBIO()
    io.set_structure(structure)
    pdb_buf = StringIO()
    io.save(pdb_buf)
    pdb_str = pdb_buf.getvalue()
    if view is None:
        view = py3Dmol.view(width=600, height=500)
    view.addModel(pdb_str, "pdb")
    return view


def add_center_line(view, start, end, color):
    """Add a line from start to end in py3Dmol view."""
    try:
        view.addLine({
            "start": {"x": float(start[0]), "y": float(start[1]), "z": float(start[2])},
            "end": {"x": float(end[0]), "y": float(end[1]), "z": float(end[2])},
            "color": color,
        })
    except AttributeError:
        view.addCylinder({
            "start": {"x": float(start[0]), "y": float(start[1]), "z": float(start[2])},
            "end": {"x": float(end[0]), "y": float(end[1]), "z": float(end[2])},
            "color": color,
            "radius": 0.1,
        })


if not PDB_DIR.exists():
    print(f"ERROR: PDB directory does not exist: {PDB_DIR}")
else:
    candidate_dirs = sorted([p for p in DATA_ROOT.iterdir() if p.is_dir()])
    target_dir = (DATA_ROOT / TARGET_ID) if TARGET_ID else (candidate_dirs[0] if candidate_dirs else None)
    if target_dir is None or not target_dir.exists():
        print(f"ERROR: Target directory not found: {target_dir}")
    else:
        missing = [fn for fn in required_files if not (target_dir / fn).exists()]
        if missing:
            print(f"ERROR: Missing required files in {target_dir.name}: {missing}")
        else:
            rmsd = np.load(target_dir / "source_patch_rmsds.npy", allow_pickle=True)
            aligned = np.load(target_dir / "aligned_source_patches.npy", allow_pickle=True)
            target_patch = np.load(target_dir / "target_patch.npy", allow_pickle=True)
            alignment_transformations = np.load(
                target_dir / "alignment_transformations.npy", allow_pickle=True
            )
            source_patch_names = np.load(
                target_dir / "source_patch_names.npy", allow_pickle=True
            )

            n = min(len(rmsd), len(aligned), len(alignment_transformations), len(source_patch_names))
            rmsd = np.asarray(rmsd[:n], dtype=float)
            aligned = aligned[:n]
            alignment_transformations = alignment_transformations[:n]
            source_patch_names = source_patch_names[:n]

            # Filter to same-complex pairs only (target and source in same coordinate frame)
            same_complex = np.array([str(source_patch_names[i]) == target_dir.name for i in range(n)])
            idx_same = np.where(same_complex)[0]
            neg_idx = np.where(rmsd >= RMSD_POSITIVE_THRESHOLD)[0]
            pos_idx = np.where(rmsd < RMSD_POSITIVE_THRESHOLD)[0]
            pos_same = np.intersect1d(pos_idx, idx_same)
            neg_same = np.intersect1d(neg_idx, idx_same)

            rng = np.random.default_rng(RNG_SEED)
            pos_show = rng.choice(pos_same, size=min(N_MAX_POS, len(pos_same)), replace=False) if len(pos_same) > 0 else np.array([], dtype=int)
            neg_show = rng.choice(neg_same, size=min(N_MAX_NEG, len(neg_same)), replace=False) if len(neg_same) > 0 else np.array([], dtype=int)

            if len(pos_show) == 0 and len(neg_show) == 0:
                print("No same-complex positive or negative pairs available to plot.")
            else:
                tgt_parts = target_dir.name.split("_")
                tgt_pdb_id = tgt_parts[0]
                tgt_chain_p1 = tgt_parts[1] if len(tgt_parts) >= 2 else tgt_parts[0]
                tgt_chain_p2 = tgt_parts[2] if len(tgt_parts) >= 3 else tgt_parts[1]
                pdb_p1 = PDB_DIR / f"{tgt_pdb_id}_{tgt_chain_p1}.pdb"
                pdb_p2 = PDB_DIR / f"{tgt_pdb_id}_{tgt_chain_p2}.pdb"

                if not pdb_p1.exists() or not pdb_p2.exists():
                    print(f"ERROR: PDB not found: {pdb_p1} or {pdb_p2}")
                else:
                    parser = PDBParser(QUIET=True)
                    struct_p1 = parser.get_structure("p1", str(pdb_p1))
                    struct_p2 = parser.get_structure("p2", str(pdb_p2))

                    target_center = np.mean(target_patch, axis=0)

                    view = py3Dmol.view(width=600, height=500)
                    add_struct_to_py3dmol(struct_p1, view)
                    view.setStyle({"model": 0}, {"cartoon": {"color": "cyan"}})
                    add_struct_to_py3dmol(struct_p2, view)
                    view.setStyle({"model": 1}, {"cartoon": {"color": "lightgreen"}})

                    # Target center sphere
                    view.addSphere({
                        "center": {"x": float(target_center[0]), "y": float(target_center[1]), "z": float(target_center[2])},
                        "radius": SPHERE_RADIUS,
                        "color": "navy",
                    })

                    for i in pos_show:
                        T = np.asarray(alignment_transformations[i])
                        pts = np.asarray(aligned[i])
                        pts_h = np.hstack([pts, np.ones((len(pts), 1))])
                        src_orig = (np.linalg.inv(T) @ pts_h.T).T[:, :3]
                        source_center = np.mean(src_orig, axis=0)
                        view.addSphere({
                            "center": {"x": float(source_center[0]), "y": float(source_center[1]), "z": float(source_center[2])},
                            "radius": SPHERE_RADIUS,
                            "color": "blue",
                        })
                        add_center_line(view, target_center, source_center, "blue")

                    for i in neg_show:
                        T = np.asarray(alignment_transformations[i])
                        pts = np.asarray(aligned[i])
                        pts_h = np.hstack([pts, np.ones((len(pts), 1))])
                        src_orig = (np.linalg.inv(T) @ pts_h.T).T[:, :3]
                        source_center = np.mean(src_orig, axis=0)
                        view.addSphere({
                            "center": {"x": float(source_center[0]), "y": float(source_center[1]), "z": float(source_center[2])},
                            "radius": SPHERE_RADIUS,
                            "color": "red",
                        })
                        add_center_line(view, target_center, source_center, "red")

                    view.zoomTo()
                    view.show()

                    print(f"P1: PDB ID {tgt_pdb_id}, chain {tgt_chain_p1}")
                    print(f"P2: PDB ID {tgt_pdb_id}, chain {tgt_chain_p2}")
                    print(f"Showing {len(pos_show)} positive, {len(neg_show)} negative pairs (same complex only).")


### Step 2.2: Precompute Features

Extract 4 features for neural network input:
```bash
sbatch prepare_features_12A.slurm
```

**What this does:**
- For each alignment, compute:
  1. `1/distance` - Spatial proximity
  2. `1/descriptor_distance` - Descriptor similarity
  3. `normal_dot_product` - Surface orientation
  4. `1/vertex_to_atom_distance` - Atom proximity
- Save as `features.npy`

**Outputs:**
```
training_data_12A_seed_benchmark/PDBID_CHAIN1_CHAIN2/
└── features.npy  # Shape: (n_alignments, n_points, 4)
```

In [ ]:
!sbatch prepare_features_12A.slurm

### Step 2.3: Train AlignmentEvaluationNN

Train the alignment scoring network:
```bash
sbatch train_nn.slurm
```

**Training configuration:**
- Network: PointNet-style (1D Conv + Global Pooling)
- Task: Binary classification (RMSD < 2.0 Å = positive)
- Input: (max_npoints=200, n_features=4)
- Output: Probability of good alignment
- Class ratio: 1 positive : 100 negatives
- Epochs: 50
- Duration: ~2-4 hours

**Outputs:**
- `models_std/weights_12A_0123.*` - Trained AlignmentEvaluationNN weights

In [ ]:
!sbatch train_nn.slurm